# 01 — Data Preparation and 70/20/10 Client Split

Loads raw tables, applies exclusion rules, persists time-window metadata, and produces the sacred 70/20/10 client split.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

In [2]:
import json
from datetime import timezone
from datetime import timedelta

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

import config

## 1. Helper functions

In [3]:
def parse_datetime(s):
    """Parse timestamp columns with trailing slash: 'd/m/yyyy/ HH:MM:SS'."""
    s = s.astype(str).str.strip().str.replace(r'(\d{4})/', r'\1', regex=True)
    return pd.to_datetime(s, format='%d/%m/%Y %H:%M:%S', errors='coerce', utc=True)


def parse_date_only(s):
    """Parse date-only columns with trailing slash: 'd/m/yyyy/'."""
    s = s.astype(str).str.strip().str.replace(r'(\d{4})/', r'\1', regex=True)
    return pd.to_datetime(s, format='%d/%m/%Y', errors='coerce', utc=True)


def read_transactions(path):
    """Read TRANSAKCIJE CSV handling the extra phantom semicolon column."""
    cols = [
        'IDENTIFIKATOR_PROIZVODA',
        'DATUM_I_VRIJEME_TRANSAKCIJE',
        'IZNOS_TRANSAKCIJE_U_DOMICILNOJ_VALUTI',
        'VALUTA_TRANSAKCIJE',
        'KANAL',
        'SMJER',
        'DRZAVA_DRUGE_STRANE',
        'DJELATNOST_DRUGE_STRANE',
        'KATEGORIJA_DJELATNOSTI_DRUGE_STRANE',
        'VRSTA_TRANSAKCIJE',
    ]
    df = pd.read_csv(
        path,
        sep=';',
        encoding='latin-1',
        names=cols + ['EXTRA'],
        header=0,
        low_memory=False,
    )
    # Fix rows where an extra semicolon shifts KATEGORIJA into VRSTA and VRSTA into EXTRA
    mask = df['EXTRA'].notna()
    df.loc[mask, 'KATEGORIJA_DJELATNOSTI_DRUGE_STRANE'] = (
        df.loc[mask, 'KATEGORIJA_DJELATNOSTI_DRUGE_STRANE'].astype(str)
        + ';'
        + df.loc[mask, 'VRSTA_TRANSAKCIJE'].astype(str)
    )
    df.loc[mask, 'VRSTA_TRANSAKCIJE'] = df.loc[mask, 'EXTRA']
    df = df.drop(columns=['EXTRA'])
    return df

## 2. Load all 4 raw tables

In [4]:
RAW = config.RAW_DATA_DIR

# --- KLIJENTI ---
clients = pd.read_csv(RAW / 'HACKATHON ZICER 202604 KLIJENTI.csv', sep=';', encoding='latin-1', low_memory=False)
for col in ['DATUM_PRVOG_POCETKA_POSLOVNOG_ODNOSA', 'DATUM_ZADNJEG_POCETKA_POSLOVNOG_ODNOSA']:
    clients[col] = parse_date_only(clients[col])
print(f'Clients:  {clients.shape}')

# --- PROIZVODI ---
products = pd.read_csv(RAW / 'HACKATHON ZICER 202604 PROIZVODI.csv', sep=';', encoding='latin-1', low_memory=False)
for col in ['DATUM_OTVARANJA', 'DATUM_ZATVARANJA']:
    products[col] = parse_date_only(products[col])
print(f'Products: {products.shape}')

# --- TRANSAKCIJE ---
tx = read_transactions(RAW / 'HACKATHON ZICER 202604 TRANSAKCIJE.csv')
tx['DATUM_I_VRIJEME_TRANSAKCIJE'] = parse_datetime(tx['DATUM_I_VRIJEME_TRANSAKCIJE'])
tx['IZNOS_TRANSAKCIJE_U_DOMICILNOJ_VALUTI'] = (
    tx['IZNOS_TRANSAKCIJE_U_DOMICILNOJ_VALUTI']
    .astype(str)
    .str.replace(',', '.', regex=False)
    .pipe(pd.to_numeric, errors='coerce')
)
print(f'Txns:     {tx.shape}')

# --- STANJA PROIZVODA ---
balances = pd.read_csv(RAW / 'HACKATHON ZICER 202604 STANJA PROIZVODA.csv', sep=';', encoding='latin-1', low_memory=False)
balances['STANJE_U_DOMICILNOJ_VALUTI'] = (
    balances['STANJE_U_DOMICILNOJ_VALUTI']
    .astype(str)
    .str.replace(',', '.', regex=False)
    .pipe(pd.to_numeric, errors='coerce')
)
print(f'Balances: {balances.shape}')

Clients:  (11997, 38)
Products: (58703, 13)
Txns:     (1187661, 10)
Balances: (817933, 6)


## 3. Compute time windows

In [5]:
valid_tx_dates = tx['DATUM_I_VRIJEME_TRANSAKCIJE'].dropna()

T0 = valid_tx_dates.min()
T_end = valid_tx_dates.max()
feature_window_end = T_end - timedelta(days=config.OUTCOME_WINDOW_DAYS)
outcome_window_start = feature_window_end  # same point

print(f'T0                 : {T0}')
print(f'T_end              : {T_end}')
print(f'feature_window_end : {feature_window_end}')
print(f'outcome_window_days: {config.OUTCOME_WINDOW_DAYS}')

T0                 : 2024-06-02 23:25:39+00:00
T_end              : 2026-04-14 23:55:04+00:00
feature_window_end : 2025-12-15 23:55:04+00:00
outcome_window_days: 120


## 4. Save time_split.json

In [6]:
config.PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

time_split = {
    'T0': T0.isoformat(),
    'T_end': T_end.isoformat(),
    'feature_window_end': feature_window_end.isoformat(),
    'outcome_window_days': config.OUTCOME_WINDOW_DAYS,
}

time_split_path = config.PROCESSED_DATA_DIR / 'time_split.json'
with open(time_split_path, 'w') as f:
    json.dump(time_split, f, indent=2)

print(f'Saved: {time_split_path}')
print(json.dumps(time_split, indent=2))

Saved: /Users/Max/Desktop/Fintech-hackathon/data/processed/time_split.json
{
  "T0": "2024-06-02T23:25:39+00:00",
  "T_end": "2026-04-14T23:55:04+00:00",
  "feature_window_end": "2025-12-15T23:55:04+00:00",
  "outcome_window_days": 120
}


## 5. Apply client exclusion rules

In [7]:
total_clients = len(clients)
print(f'Total clients loaded: {total_clients}')

eligible = clients.copy()

# Rule 1: Drop deceased clients
# The column uses 'Y'/'N' (English) in this dataset; also handle 'D'/'DA' (Croatian)
deceased_mask = eligible['KLIJENT_PREMINUO'].astype(str).str.strip().str.upper().isin(['1', 'D', 'TRUE', 'T', 'DA', 'Y'])
n_deceased = deceased_mask.sum()
eligible = eligible[~deceased_mask].copy()
print(f'Dropped (deceased)         : {n_deceased}')

# Rule 2: Drop KYC-blocked clients
# Same encoding: 'Y' means blocked
kyc_mask = eligible['IMA_LI_KLIJENT_KYC_BLOKADU'].astype(str).str.strip().str.upper().isin(['1', 'D', 'TRUE', 'T', 'DA', 'Y'])
n_kyc = kyc_mask.sum()
eligible = eligible[~kyc_mask].copy()
print(f'Dropped (KYC blocked)      : {n_kyc}')

# Rule 3: Drop clients with fewer than MIN_TX_FEATURE_WINDOW transactions in feature window
# Join tx -> products to get client IDs, then filter to feature window
tx_feature = tx[
    tx['DATUM_I_VRIJEME_TRANSAKCIJE'].notna() &
    (tx['DATUM_I_VRIJEME_TRANSAKCIJE'] >= T0) &
    (tx['DATUM_I_VRIJEME_TRANSAKCIJE'] < feature_window_end)
].copy()

# Map product -> client
prod_to_client = products[['IDENTIFIKATOR_PROIZVODA', 'IDENTIFIKATOR_KLIJENTA']].drop_duplicates()
tx_feature = tx_feature.merge(prod_to_client, on='IDENTIFIKATOR_PROIZVODA', how='left')

tx_count_feature = (
    tx_feature.groupby('IDENTIFIKATOR_KLIJENTA')
    .size()
    .rename('tx_count_feature_window')
)

eligible = eligible.merge(tx_count_feature, on='IDENTIFIKATOR_KLIJENTA', how='left')
eligible['tx_count_feature_window'] = eligible['tx_count_feature_window'].fillna(0)

few_tx_mask = eligible['tx_count_feature_window'] < config.MIN_TX_FEATURE_WINDOW
n_few_tx = few_tx_mask.sum()
eligible = eligible[~few_tx_mask].copy()
print(f'Dropped (< {config.MIN_TX_FEATURE_WINDOW} tx in feature window): {n_few_tx}')

# Rule 4: Drop clients whose earliest product opening is <120 days before T_end (too new)
earliest_product = (
    products.groupby('IDENTIFIKATOR_KLIJENTA')['DATUM_OTVARANJA']
    .min()
    .rename('earliest_product_date')
)
eligible = eligible.merge(earliest_product, on='IDENTIFIKATOR_KLIJENTA', how='left')

# Ensure T_end is tz-aware for comparison; earliest_product_date is tz-aware (utc=True)
too_new_mask = (T_end - eligible['earliest_product_date']).dt.days < config.OUTCOME_WINDOW_DAYS
n_too_new = too_new_mask.sum()
eligible = eligible[~too_new_mask].copy()
print(f'Dropped (product < {config.OUTCOME_WINDOW_DAYS}d before T_end): {n_too_new}')

print(f'\nTotal eligible clients     : {len(eligible)}')

Total clients loaded: 11997
Dropped (deceased)         : 345
Dropped (KYC blocked)      : 135


Dropped (< 10 tx in feature window): 6903
Dropped (product < 120d before T_end): 0

Total eligible clients     : 4614


## 6. Perform (or load) 70/20/10 split

In [8]:
split_train_path   = config.PROCESSED_DATA_DIR / 'split_train.json'
split_test_path    = config.PROCESSED_DATA_DIR / 'split_test.json'
split_holdout_path = config.PROCESSED_DATA_DIR / 'split_holdout.json'

all_exist = split_train_path.exists() and split_test_path.exists() and split_holdout_path.exists()

if all_exist:
    print('Split files already exist — loading from disk (NOT regenerating).')
    with open(split_train_path)   as f: train_ids   = json.load(f)
    with open(split_test_path)    as f: test_ids    = json.load(f)
    with open(split_holdout_path) as f: holdout_ids = json.load(f)
else:
    print('Split files not found — generating fresh split with random_state={}.'.format(config.RANDOM_STATE))
    all_ids = eligible['IDENTIFIKATOR_KLIJENTA'].tolist()

    # First split: 90% dev, 10% holdout
    dev_ids, holdout_ids = train_test_split(
        all_ids,
        test_size=config.SPLIT_HOLDOUT,
        random_state=config.RANDOM_STATE,
    )

    # Second split: from dev, produce train (~70%) and test (~20%) of total
    # test_size = SPLIT_TEST / (1 - SPLIT_HOLDOUT) so final ratios sum to 100%
    adjusted_test_size = config.SPLIT_TEST / (1.0 - config.SPLIT_HOLDOUT)
    train_ids, test_ids = train_test_split(
        dev_ids,
        test_size=adjusted_test_size,
        random_state=config.RANDOM_STATE,
    )

    # Save
    with open(split_train_path,   'w') as f: json.dump(train_ids,   f)
    with open(split_test_path,    'w') as f: json.dump(test_ids,    f)
    with open(split_holdout_path, 'w') as f: json.dump(holdout_ids, f)
    print(f'Saved: {split_train_path}')
    print(f'Saved: {split_test_path}')
    print(f'Saved: {split_holdout_path}')

print(f'\nTrain  : {len(train_ids):,}')
print(f'Test   : {len(test_ids):,}')
print(f'Holdout: {len(holdout_ids):,}')
print(f'Total  : {len(train_ids) + len(test_ids) + len(holdout_ids):,}')

# Verify disjoint
assert len(set(train_ids) & set(test_ids)) == 0,    'Train/test overlap!'
assert len(set(train_ids) & set(holdout_ids)) == 0, 'Train/holdout overlap!'
assert len(set(test_ids)  & set(holdout_ids)) == 0, 'Test/holdout overlap!'
print('\nDisjoint check: PASSED')

Split files already exist — loading from disk (NOT regenerating).

Train  : 3,126
Test   : 1,026
Holdout: 462
Total  : 4,614

Disjoint check: PASSED


## 7. Save clients_eligible.parquet with split column

In [9]:
# Build split lookup
split_map = {}
for cid in train_ids:   split_map[cid] = 'train'
for cid in test_ids:    split_map[cid] = 'test'
for cid in holdout_ids: split_map[cid] = 'holdout'

# Only keep eligible clients that appear in one of the three splits
eligible['split'] = eligible['IDENTIFIKATOR_KLIJENTA'].map(split_map)

# Drop helper columns added during exclusion
eligible = eligible.drop(columns=['tx_count_feature_window', 'earliest_product_date'], errors='ignore')

clients_eligible_path = config.PROCESSED_DATA_DIR / 'clients_eligible.parquet'
eligible.to_parquet(clients_eligible_path, index=False)
print(f'Saved: {clients_eligible_path}')
print(f'Shape: {eligible.shape}')
print(f"Split value counts:\n{eligible['split'].value_counts()}")

Saved: /Users/Max/Desktop/Fintech-hackathon/data/processed/clients_eligible.parquet
Shape: (4614, 39)
Split value counts:
split
train      3126
test       1026
holdout     462
Name: count, dtype: int64


## 8. Summary

In [10]:
print('=' * 55)
print('DATA PREPARATION SUMMARY')
print('=' * 55)
print(f'Total clients loaded                  : {total_clients:,}')
print(f'  Dropped (deceased)                  : {n_deceased:,}')
print(f'  Dropped (KYC blocked)               : {n_kyc:,}')
print(f'  Dropped (< {config.MIN_TX_FEATURE_WINDOW} tx in feature window) : {n_few_tx:,}')
print(f'  Dropped (product < {config.OUTCOME_WINDOW_DAYS}d before T_end): {n_too_new:,}')
print(f'Total eligible                        : {len(eligible):,}')
print()
print(f'Split sizes:')
print(f'  Train   : {len(train_ids):,} ({len(train_ids)/len(eligible)*100:.1f}%)')
print(f'  Test    : {len(test_ids):,} ({len(test_ids)/len(eligible)*100:.1f}%)')
print(f'  Holdout : {len(holdout_ids):,} ({len(holdout_ids)/len(eligible)*100:.1f}%)')
print()
print(f'Time windows:')
print(f'  T0                 : {T0}')
print(f'  T_end              : {T_end}')
print(f'  feature_window_end : {feature_window_end}')
print('=' * 55)

DATA PREPARATION SUMMARY
Total clients loaded                  : 11,997
  Dropped (deceased)                  : 345
  Dropped (KYC blocked)               : 135
  Dropped (< 10 tx in feature window) : 6,903
  Dropped (product < 120d before T_end): 0
Total eligible                        : 4,614

Split sizes:
  Train   : 3,126 (67.8%)
  Test    : 1,026 (22.2%)
  Holdout : 462 (10.0%)

Time windows:
  T0                 : 2024-06-02 23:25:39+00:00
  T_end              : 2026-04-14 23:55:04+00:00
  feature_window_end : 2025-12-15 23:55:04+00:00


## 9. Distinct KATEGORIJA_DJELATNOSTI_DRUGE_STRANE values

In [11]:
categories = (
    tx['KATEGORIJA_DJELATNOSTI_DRUGE_STRANE']
    .dropna()
    .str.strip()
    .unique()
)
categories_sorted = sorted(categories)

print(f'Distinct KATEGORIJA_DJELATNOSTI_DRUGE_STRANE values ({len(categories_sorted)} total):')
print()
for cat in categories_sorted:
    print(f'  {cat}')

Distinct KATEGORIJA_DJELATNOSTI_DRUGE_STRANE values (34 total):

  ADMINISTRATIVNE I POMOÆNE USLUNE DJELATNOSTI
  Agricultural services
  Amusement and entertainment
  Business services
  Clothing outlets
  Contracted services
  DJELATNOSTI PRUANJA SMJETAJA TE PRIPREME I USLUIVANJA HRANE
  DJELATNOSTI ZDRAVSTVENE ZATITE I SOCIJALNE SKRBI
  Error
  FINANCIJSKE DJELATNOSTI I DJELATNOSTI OSIGURANJA
  GRAÐEVINARSTVO
  Government services
  INFORMACIJE I KOMUNIKACIJE
  JAVNA UPRAVA I OBRANA; OBVEZNO SOCIJALNO OSIGURANJE
  Miscellaneous outlets
  NEPOZNATO
  OBRAZOVANJE
  OPSKRBA ELEKTRIÈNOM ENERGIJOM, PLINOM, PAROM I KLIMATIZACIJA
  OPSKRBA VODOM; UKLANJANJE OTPADNIH VODA, GOSPODARENJE OTPADOM TE DJELATNOSTI SANACIJE OKOLIA
  OSTALE USLUNE DJELATNOSTI
  POLJOPRIVREDA, UMARSTVO I RIBARSTVO
  POSLOVANJE NEKRETNINAMA
  PRERAÐIVAÈKA INDUSTRIJA
  PRIJEVOZ I SKLADITENJE
  PROIZVODNJA SVJEIH PECIVA I SLIÈNIH PROIZVODA TE KOLAÈA                                                             